# Creating DataFrames: 5 ways

Every transformation and action in Spark starts from a DataFrame, so the first step of any job is to **create one**.
This notebook shows the 5 most common ways to do it:

| # | Way | Method | Typical use |
|---|---|---|---|
| 1A | Read a **CSV file** with a connector | `spark.read.format("csv").option(...).load(path)` | Raw files in the data lake |
| 1B | Read a **JSON file** with a connector | `spark.read.format("json").load(path)` | Raw files in the data lake |
| 2 | Read a **Spark table** | `spark.table("catalog.schema.table")` | Data already saved as a table |
| 3 | Run a **SQL query** | `spark.sql("select ...")` | You need only part of a table, or the logic is easier in SQL |
| 4 | Convert a **Python list** | `spark.createDataFrame(data, schema)` | Small test data |
| 5 | Generate a **range of numbers** | `spark.range(start, end, step)` | Quick test data, one `id` column |

> All of these are **lazy** except where we call an action (`show()`): creating the DataFrame only builds a plan.
> (Exception: `inferSchema` makes Spark scan the file once to guess the column types.)

## 1A. Read a CSV file with a connector

We already used this approach before. The requirement: read a file stored in the data lake and create a DataFrame from it.

`spark.read` gives us access to Spark's **connector framework** (the `DataFrameReader`):

1. `format("csv")`: choose the connector
2. `option(key, value)`: configure the connector (one `option` call per setting)
3. `load(path)`: point it to the file and create the DataFrame

Options used here:
- `header = true`: the first line holds the column names, not data
- `inferSchema = true`: let Spark guess the column types (otherwise every column is a `string`)

The path starts with `/Volumes/...`, which is a **Unity Catalog volume** (files stored under `catalog / schema / volume`).

In [ ]:
# CSV connector: header row = column names, guess the column types
file_df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(path="/Volumes/dev/spark_db/datasets/spark_programming/data/sf-fire-calls.csv")
)

### Every connector has its own options

Spark has connectors for many kinds of sources: CSV, JSON, Parquet, ORC, Avro, text, even binary files (images, audio, video), and databases through JDBC.

Each connector comes with **its own set of options**, and we set them with `.option(...)` after choosing the connector with `.format(...)`.
An option that means something for CSV (like `header`) doesn't exist for every other format.

## 1B. Read a JSON file with a connector

Same pattern, different connector: only the `format` changes.

We usually don't know which options a connector supports (in this case JSON), so we look them up in the documentation on the Apache Spark website:
- [JSON data source options](https://spark.apache.org/docs/latest/sql-data-sources-json.html)
- [CSV data source options](https://spark.apache.org/docs/latest/sql-data-sources-csv.html)

Notes for JSON:
- No `header` or `inferSchema` options are needed: the column names and types come from the JSON keys and values.
- By default Spark expects **one JSON object per line** (JSON Lines). For a file that holds one big JSON array or objects spread over many lines, add `.option("multiLine", "true")`.

In [ ]:
# JSON connector: column names and types come from the JSON itself
json_file_df = (
    spark.read.format("json")
         .load(path="/Volumes/dev/spark_db/datasets/spark_programming/data/diamonds.json")
)

## 2. Read a Spark table

If the data is already stored in a **Spark table**, we don't need a connector or any options:
the table already knows its location, format and schema. We just use its full name `catalog.schema.table`.

`spark.table(...)` and `spark.read.table(...)` do the same thing.

In [ ]:
# The whole table as a DataFrame: same as "select * from dev.spark_db.sf_fire_calls"
table_df = spark.table("dev.spark_db.sf_fire_calls")

## 3. Run a SQL query

Sometimes we need to **query** Spark tables (filter, join, aggregate) and create a DataFrame from the result.
`spark.sql(...)` runs any SQL query and returns its result as a DataFrame.

- The query is a normal Python string, so we can build it with an **f-string** (here, the table name comes from a variable).
- The result is a regular DataFrame: we can keep chaining DataFrame methods on it.
- `limit 5` keeps only 5 rows, and `show()` is the action that runs the query and prints them.

In [5]:
table_name = "dev.spark_db.sf_fire_calls"

# Build the query with an f-string; the result of the query is a DataFrame
sql_df = spark.sql(f"select * from {table_name} limit 5")

# Action: run the query and print the rows
sql_df.show()

+----------+------+--------------+----------------+----------+----------+--------------------+-------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------+--------------------------+----------------------+------------------+--------------------+--------------------+-------------+---------+
|CallNumber|UnitID|IncidentNumber|        CallType|  CallDate| WatchDate|CallFinalDisposition|      AvailableDtTm|             Address|City|Zipcode|Battalion|StationArea| Box|OriginalPriority|Priority|FinalPriority|ALSUnit|CallTypeGroup|NumAlarms|UnitType|UnitSequenceInCallDispatch|FirePreventionDistrict|SupervisorDistrict|        Neighborhood|            Location|        RowID|    Delay|
+----------+------+--------------+----------------+----------+----------+--------------------+-------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-

## 4. Create a DataFrame from a Python list

Mainly used for **testing Spark code**: we write some test data in a Python list and convert it into a DataFrame.

- Each **tuple** in the list is one row.
- The **schema** is a DDL string: `column_name type, column_name type, ...`.
  The values must match the types: `date(...)` for `date`, `datetime(...)` for `timestamp`, a float for `double`.
- `spark.createDataFrame(data, schema)` builds the DataFrame.

> **Why does `created_at` show 2 hours less than in the list?**
> The `datetime` values have no time zone, so Python treats them as **local time** (UTC+2 here).
> Spark stores timestamps in UTC and `show()` prints them in the **session time zone** (UTC on the cluster), so `09:00` becomes `07:00`.
> It is the same moment in time, shown in a different time zone.

In [3]:
from datetime import datetime, date

# Schema as a DDL string: "name type, name type, ..."
data_list_schema = 'id long, name string, joining_date date, salary double, created_at timestamp'

# One tuple = one row; values must match the schema's types
data_list = [(1, "Prashant", date(2018, 1, 1), 924.0, datetime(2022, 1, 1, 9, 0)),
             (2, "Sushant", date(2019, 2, 1), 1260.50, datetime(2022, 1, 2, 11, 0)),
             (3, "David", date(2022, 3, 1), 765.0, datetime(2022, 1, 3, 10, 0))]

list_df = spark.createDataFrame(data_list, data_list_schema)

list_df.show()

+---+--------+------------+------+-------------------+
| id|    name|joining_date|salary|         created_at|
+---+--------+------------+------+-------------------+
|  1|Prashant|  2018-01-01| 924.0|2022-01-01 07:00:00|
|  2| Sushant|  0201-02-01|1260.5|2022-01-02 09:00:00|
|  3|   David|  2022-03-01| 765.0|2022-01-03 08:00:00|
+---+--------+------------+------+-------------------+



## 5. Create a single-column DataFrame from a range

`spark.range(start, end, step)` creates a DataFrame with **one column named `id`** (type `long`).

- `start` is included, `end` is **excluded** (like Python's `range`): `range(1000, 1010, 2)` gives 1000, 1002, 1004, 1006, 1008, without 1010.
- With one argument, `spark.range(5)` gives 0 to 4.
- Useful to quickly generate test data, for example a lot of rows to try out a transformation.

In [4]:
# Numbers from 1000 up to (not including) 1010, step 2, in a column named "id"
range_df = spark.range(1000, 1010, 2)
range_df.show()

+----+
|  id|
+----+
|1000|
|1002|
|1004|
|1006|
|1008|
+----+

